# 3.3 线性回归的简洁实现

本节使用 PyTorch 深度学习框架的内置 API 实现线性回归，复用框架封装好的数据迭代器、全连接层、损失函数与优化器，大幅简化重复代码，核心训练逻辑与 3.2 节从零实现完全一致。

## 3.3.1 生成数据集
与3.2节数据生成逻辑一致，这里直接调用d2l库封装好的synthetic_data函数生成带噪声的线性数据集。真实权重 $w=[2,-3.4]^\top$，偏置 $b=4.2$，共生成1000个样本，每个样本包含2个特征。


In [ ]:
%matplotlib inline
import torch
from torch.utils import data
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

## 3.3.2 读取数据集
使用PyTorch原生数据接口构建高效数据迭代器：
1.	TensorDataset 将特征矩阵与标签向量封装为标准数据集对象；
2.	DataLoader 按指定批量大小切分数据，支持自动打乱顺序，底层优化了内存访问效率，性能优于手写迭代器。

is_train 参数控制是否在每个迭代周期打乱数据，训练阶段设为True以增强样本随机性。


In [ ]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

验证迭代器功能，读取并打印第一个小批量样本。

In [ ]:
next(iter(data_iter))

## 3.3.3 定义模型
使用 nn.Sequential 作为层容器，按顺序串联网络层；线性回归对应单个全连接层（nn.Linear），输入特征维度为2，输出标签维度为1。

框架会自动管理该层的权重与偏置参数，无需手动声明与追踪梯度，功能等价于3.2节手动实现的linreg函数。


In [ ]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

## 3.3.4 初始化模型参数
通过 net[0] 访问Sequential容器中的第一个线性层，使用weight.data与bias.data修改参数数值：
- 权重从均值为0、标准差为0.01的正态分布中采样
- 偏置初始化为0

初始化规则与3.2节从零实现完全一致。


In [ ]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

## 3.3.5 定义损失函数
使用PyTorch内置的均方误差损失 nn.MSELoss，默认返回批量样本的平均损失，公式为： $$L = \frac{1}{B}\sum_{i=1}^B (\hat{y}_i - y_i)^2$$ 其中 $B$ 为批量大小，对应3.2节平方损失的平均版本。


In [ ]:
loss = nn.MSELoss()

## 3.3.6 定义优化算法
使用 torch.optim.SGD 实现小批量随机梯度下降，传入模型全部可训练参数与学习率 lr=0.03。 优化器会自动完成参数更新，替代3.2节手写的sgd函数。


In [ ]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 3.3.7 模型训练
训练循环逻辑与从零实现一致，每个小批量迭代包含三步：
1.	前向传播：调用net(X)计算预测值，代入损失函数得到批量损失
2.	反向传播：先清零梯度缓存，再对损失执行反向传播，计算参数梯度
3.	参数更新：调用优化器的step()方法，按梯度更新模型全部参数

每个迭代周期（epoch）遍历完整数据集后，计算全量训练集的平均损失并输出。


In [ ]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

**参数效果评估**

从训练好的全连接层中取出权重与偏置，与真实参数对比误差，验证模型的学习效果。


In [ ]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)